# AAPL Monte Carlo Simulation

In [74]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import yfinance as yf

data = yf.download("NVDA", period="2y")
data.to_csv("NVDA_2Y_OHLC.csv", index_label="Date")

data = yf.download("AAPL", period="2y")
data.to_csv("AAPL_2Y_OHLC.csv", index_label="Date")

np.random.seed(42)

AAPL_DATA_PATH = Path("AAPL_2Y_OHLC.csv")
STEPS = 90
N_SIMS = 2000
CONFIDENCE_LEVEL = [0.05, 0.25, 0.5, 0.75, 0.95]
VAR_LEVEL = 0.05
PLOT_PATHS = 500 

print('Config loaded successfully.')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Config loaded successfully.


## 1. Load & Inspect Historocal Data

In [78]:
aapl_raw = pl.read_csv(
    'AAPL_2Y_OHLC.csv',
    skip_rows=3,
    has_header=False,
    new_columns=['datetime', 'close', 'high', 'low', 'open', 'volume']
)

# Parse datetime (has timezone offset +00:00) and reorder to match BTC format
aapl = (
    aapl_raw
    .with_columns(
        pl.col('datetime')
          .str.to_datetime(format='%Y-%m-%d', time_unit='us')
          .alias('datetime')
    )
    .select(['datetime', 'open', 'high', 'low', 'close'])  # drop volume, reorder
    .sort('datetime')
)

print(aapl.head())
aapl.write_csv('AAPL_2Y_ohlc_clean.csv')

shape: (5, 5)
┌─────────────────────┬────────────┬────────────┬────────────┬────────────┐
│ datetime            ┆ open       ┆ high       ┆ low        ┆ close      │
│ ---                 ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ datetime[μs]        ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞═════════════════════╪════════════╪════════════╪════════════╪════════════╡
│ 2024-03-21 00:00:00 ┆ 175.453229 ┆ 175.889263 ┆ 169.299229 ┆ 169.824448 │
│ 2024-03-22 00:00:00 ┆ 170.210958 ┆ 171.489332 ┆ 168.526293 ┆ 170.726273 │
│ 2024-03-25 00:00:00 ┆ 169.03167  ┆ 170.389309 ┆ 167.92176  ┆ 169.309143 │
│ 2024-03-26 00:00:00 ┆ 168.466806 ┆ 169.873998 ┆ 168.050596 ┆ 168.179428 │
│ 2024-03-27 00:00:00 ┆ 168.873109 ┆ 172.034341 ┆ 168.575811 ┆ 171.746948 │
└─────────────────────┴────────────┴────────────┴────────────┴────────────┘


In [79]:
df = pl.read_csv(AAPL_DATA_PATH, try_parse_dates = True)
print(f'Loaded {len(df):,} bars  |  {df.min()}  →  {df.max()}')
df.head(5)


Loaded 503 bars  |  shape: (1, 6)
┌────────────┬──────────────────┬──────────────────┬─────────────────┬─────────────────┬───────────┐
│ Price      ┆ Close            ┆ High             ┆ Low             ┆ Open            ┆ Volume    │
│ ---        ┆ ---              ┆ ---              ┆ ---             ┆ ---             ┆ ---       │
│ str        ┆ str              ┆ str              ┆ str             ┆ str             ┆ str       │
╞════════════╪══════════════════╪══════════════════╪═════════════════╪═════════════════╪═══════════╡
│ 2024-03-21 ┆ 163.511871337890 ┆ 164.899238743198 ┆ 162.60017059709 ┆ 163.85875133255 ┆ 100959800 │
│            ┆ 62               ┆ 42               ┆ 256             ┆ 257             ┆           │
└────────────┴──────────────────┴──────────────────┴─────────────────┴─────────────────┴───────────┘  →  shape: (1, 6)
┌────────┬───────┬──────┬──────┬──────┬────────┐
│ Price  ┆ Close ┆ High ┆ Low  ┆ Open ┆ Volume │
│ ---    ┆ ---   ┆ ---  ┆ ---  ┆ ---  ┆ --

Price,Close,High,Low,Open,Volume
str,str,str,str,str,str
"""Ticker""","""AAPL""","""AAPL""","""AAPL""","""AAPL""","""AAPL"""
"""Date""",null,null,null,null,null
"""2024-03-21""","""169.82444763183594""","""175.8892630091755""","""169.29922879245808""","""175.45322885093574""","""106181300"""
"""2024-03-22""","""170.7262725830078""","""171.48933248689517""","""168.5262926884995""","""170.21095802650942""","""71160100"""
"""2024-03-25""","""169.30914306640625""","""170.3893089389317""","""167.92176032191367""","""169.03166954174233""","""54288300"""


In [80]:
df = pl.read_csv(AAPL_DATA_PATH, try_parse_dates = True)
print(f'Loaded {len(df):,} bars  |  {df["Date"].min()}  →  {df["Date"].max()}')
df.head(5)

ColumnNotFoundError: "Date" not found